# NLP: Customer Voice and Support Sentiment

This notebook runs Transformer-based sentiment scoring and optional TF-IDF + NMF topic discovery on customer support text. The default sentiment checkpoint is general English SST-2 sentiment, so treat its scores as exploratory until evaluated against labelled telecom support messages.

## Expected data

Place a CSV at `data/raw/customer_support_text.csv` with `customerID`, `ticket_text`, and preferably `ticket_date`. Use properly licensed or consented data, protect customer identifiers, and do not commit real ticket text to source control.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "raw" / "customer_support_text.csv"
if not DATA_PATH.exists():
    DATA_PATH = ROOT / "data" / "demo" / "customer_support_text_demo.csv"
if DATA_PATH.exists():
    tickets = pd.read_csv(DATA_PATH)
    display(tickets.head())
    print("Rows:", len(tickets), "Unique customers:", tickets["customerID"].nunique())
else:
    print(f"Create the support text file first: {DATA_PATH}")

## Run sentiment and topic discovery

The first run downloads model weights from Hugging Face and requires network access. The CLI is recommended for repeatable runs; the notebook code below offers an interactive alternative.

In [ ]:
from src.nlp import SupportSentimentAnalyzer, analyze_support_tickets

if DATA_PATH.exists():
    analyzer = SupportSentimentAnalyzer(batch_size=16)
    scored_tickets, customer_features, topic_artifacts = analyze_support_tickets(
        tickets,
        id_column="customerID",
        text_column="ticket_text",
        date_column="ticket_date" if "ticket_date" in tickets.columns else None,
        analyzer=analyzer,
        n_topics=5,
    )
    display(scored_tickets.head())
    display(customer_features.head())
    if topic_artifacts is not None:
        display(pd.DataFrame(topic_artifacts["topic_info"]))

## Review sentiment and topics

Read sample tickets from each sentiment bucket and review top words / examples for every NMF topic. Topics are corpus-specific, unsupervised groupings and do not have predefined business meaning. Sentiment is not ground truth. Before using the results in churn prediction, validate them on a labelled sample and ensure all messages predate the prediction cutoff.

In [ ]:
if DATA_PATH.exists():
    display(scored_tickets["sentiment_label"].value_counts(dropna=False).rename_axis("Sentiment").to_frame("Tickets"))
    if "topic_label" in scored_tickets.columns:
        display(scored_tickets.groupby("topic_label", dropna=False).size().sort_values(ascending=False).rename("Tickets").to_frame())

    OUTPUT_DIR = ROOT / "data" / "processed" / "customer_voice"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    scored_tickets.to_csv(OUTPUT_DIR / "customer_tickets_scored.csv", index=False)
    customer_features.to_csv(OUTPUT_DIR / "customer_text_features.csv", index=False)
    print("Wrote aggregate customer features to", OUTPUT_DIR / "customer_text_features.csv")

## Validation checklist

1. Manually label a representative sample of real support messages.
2. Use `python -m scripts.evaluate_nlp_sentiment --input PATH_TO_LABELLED_CSV` to compute classification metrics.
3. Inspect false positives, false negatives, slang, short messages, multilingual text, and sarcasm.
4. Do not treat a sentiment confidence score as churn probability.
5. Align all messages to a prediction date before merging sentiment aggregates with churn labels.